# 02 – 2R2C-Modell je Geschoss

Two-floor grey-box model (`rc_house.models.two_floor_2r2c`): per floor an air node `Ti` (measured,
mean of the room thermostats) and an envelope/mass node `Te` (hidden). Couplings: Ti–Te, Te–outside,
Ti–outside, Ti_og–Ti_eg (ceiling), Ti_eg–ground (constant `T_erdreich`). Inputs: outside temperature,
global radiation and heat per circuit.

**Heat input** – `HEAT = "measured"`: Q = ṁ c_p (T_VL − T_RL) · pump duty, gain `k_Q` ≈ 1 absorbs the
error in ṁ. `HEAT = "duty"`: only the pump duty, gain `k_P` = mean radiator power while the pump runs.

Run `01_daten.ipynb` first (refreshes `data/house_15min.csv`). The fit needs at least `MIN_DAYS`
contiguous days with all inputs (GeoSphere and VL/RL only since 2026-10-13).

In [ ]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from rc_house import prep
from rc_house.models import two_floor_2r2c

DATA = Path("../data")
HEAT = "measured"      # "measured" | "duty"
MIN_DAYS = 5
T_NORM = -14.0         # design outside temperature Weyer [°C] - TODO: check ÖNORM B 8135
T_IN = 20.0            # design indoor temperature [°C]
# room weights for the floor mean (e.g. heated area in m²); empty = equal weights
WEIGHTS = {}

## Daten

In [ ]:
df = pd.read_csv(DATA / "house_15min.csv", index_col=0, parse_dates=True).asfreq("15min")
md_ = prep.model_data(df, weights=WEIGHTS)
print("non-NaN rows per column:"); print(md_.notna().sum().to_string())
fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
md_[["T_og", "T_eg", "T_aussen"]].plot(ax=ax[0]); ax[0].set_ylabel("°C")
md_[["Q_og", "Q_eg"]].plot(ax=ax[1]); ax[1].set_ylabel("Wärme [W]")
md_[["pump_og", "pump_eg"]].plot(ax=ax[2]); ax[2].set_ylabel("Pumpe [-]")
plt.tight_layout()

## Identifikation

In [ ]:
net, p0 = two_floor_2r2c(HEAT)
use = net.usable(md_)
days = len(use) / 96
print(f"usable block: {use.index[0]} .. {use.index[-1]} ({days:.1f} d)")
assert days >= MIN_DAYS, f"only {days:.1f} d of contiguous data - collect more before fitting"

# Start with the envelope split and the ground fixed; free them once the data covers
# a range of outside temperatures (identifiability).
FIXED = {k: p0[k] for k in ["C_Te_og", "C_Te_eg", "H_ie_og", "H_ie_eg", "T_erdreich", "H_g"]}
fit = net.fit(use, p0, fixed=FIXED)
print(f"RMSE {fit.rmse:.3f} K, {fit.ls.message}")
fit.summary()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for f, c in [("og", "C0"), ("eg", "C1")]:
    ax[0].plot(use.index, use[f"T_{f}"], c, lw=0.8, label=f"gemessen {f.upper()}")
    ax[0].plot(fit.sim.index, fit.sim[f"Ti_{f}"], c + "--", lw=1, label=f"Modell {f.upper()}")
    ax[1].plot(fit.sim.index, fit.sim[f"Te_{f}"], c, lw=1, label=f"Te {f.upper()} (versteckt)")
    ax[2].plot(fit.residuals.index, fit.residuals[f"Ti_{f}"], c, lw=0.6, label=f"Residuum {f.upper()}")
ax[0].plot(use.index, use["T_aussen"], "k", lw=0.6, label="außen")
for a, l in zip(ax, ["Raumluft [°C]", "Hülle [°C]", "Residuum [K]"]):
    a.set_ylabel(l); a.legend(ncol=5, fontsize=8)
plt.tight_layout()

## Heizlast bei Norm-Außentemperatur

In [ ]:
q = net.heat_load(fit.params, T_in=T_IN, T_out={"T_aussen": T_NORM})
print(f"Heizlast bei {T_NORM} °C außen / {T_IN} °C innen:")
print((q / 1000).round(2).rename(lambda s: s.replace("Ti_", "").upper()).to_string(), "kW")
print(f"gesamt: {q.sum() / 1000:.2f} kW (ohne interne/solare Gewinne)")

## Kalibrierung ṁ über Ölverbrauch (offen)

∫(Q_og + Q_eg) dt over a period ≈ oil consumed × 10 kWh/L × η (η ≈ 0.80–0.85), together with burner
hours. Needs the manual `input_number` readings (tank levels, burner hours), which are **not** in
Influx yet - add them to the HA InfluxDB `include` filter.

In [ ]:
E = use[["Q_og", "Q_eg"]].sum() * 900 / 3.6e6  # kWh over the fit window (with ṁ initial guess)
print(E.round(1).to_string(), "kWh")

In [ ]:
out = DATA / f"params_2r2c_{HEAT}_{use.index[-1]:%Y%m%d}.json"
out.write_text(json.dumps({"window": [str(use.index[0]), str(use.index[-1])], "rmse": fit.rmse,
                           "params": fit.table["value"].to_dict(), "sigma": fit.table["sigma"].dropna().to_dict(),
                           "heat_load_W": q.to_dict(), "T_norm": T_NORM}, indent=2))
out